# 第 5 周练习：专家知识工作者（RAG）

本笔记本为保险科技公司 **Insurellm** 实现一条完整的 **RAG（检索增强生成）** 管道，串起第 5 周五天的核心概念：

- **第 1 天**：基于关键字的简单上下文检索 RAG
- **第 2 天**：文档分块（Chunking）、向量嵌入、Chroma 存储与可视化
- **第 3 天**：LangChain 完整 RAG（Retriever + LLM）
- **第 4 天**：评估（检索指标 + LLM-as-judge 思路；本格用测试样本抽检）
- **第 5 天**：高级技巧（可选：重排序、查询重写——本练习主路径是标准向量 RAG）

### 运行要求
- 推荐从 **week5** 目录启动：`cd week5 && jupyter notebook`
- 也可从仓库根启动——下面的设置单元会自动 `chdir` 到 week5
- 确保 `.env` 含 `OPENAI_API_KEY`（若用 HuggingFace 嵌入，可选配置 `HF_TOKEN`）


## 设置

配置工作目录、导入依赖、加载 API Key，并选定聊天模型与向量库目录名。


In [ ]:
# ========== 把工作目录切到 week5，保证 knowledge-base / evaluation 相对路径可用 ==========
import sys
import os
from pathlib import Path

# 当前 Jupyter 的 cwd：可能是仓库根，也可能已经在 week5
repo_root = Path.cwd()
if (repo_root / "week5").exists():
    # 在仓库根 → 进入 week5
    week5_dir = repo_root / "week5"
else:
    # 已经在 week5（或等价目录）
    week5_dir = repo_root  # Already in week5
# 把 week5 插到 sys.path 最前，便于 import evaluation.*
sys.path.insert(0, str(week5_dir))
# 真正切换进程 cwd，后面 glob("knowledge-base/*") 才能命中
os.chdir(week5_dir)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# ========== 导入 LangChain / Gradio，加载密钥与模型常量 ==========
# glob：枚举 knowledge-base 子目录
import glob
# load_dotenv：读 .env 进环境变量
from dotenv import load_dotenv
# OpenAIEmbeddings：OpenAI 嵌入（本格主路径用 HF，此导入作备选）
from langchain_openai import OpenAIEmbeddings
# Chroma：持久化向量库封装
from langchain_chroma import Chroma
# HuggingFaceEmbeddings：本地/HF 小模型嵌入（免费）
from langchain_huggingface import HuggingFaceEmbeddings
# DirectoryLoader + TextLoader：按目录批量读 Markdown
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# RecursiveCharacterTextSplitter：按字符递归分块（第 2 天风格）
from langchain_text_splitters import RecursiveCharacterTextSplitter
# ChatOpenAI：聊天补全
from langchain_openai import ChatOpenAI
# SystemMessage / HumanMessage：拼 messages 列表
from langchain_core.messages import SystemMessage, HumanMessage
# Document：文档类型（加载器返回）
from langchain_core.documents import Document
# gradio：最后一格聊天 UI
import gradio as gr

# override=True：以 .env 覆盖已有环境变量
load_dotenv(override=True)

# 聊天模型 id（勿改）
MODEL = "gpt-4.1-nano"
# Chroma 持久化目录名
DB_NAME = "vector_db"

# 检查 OpenAI Key（ChatOpenAI 需要）；打印前缀便于确认已加载
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key found (starts with {openai_api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set — add it to .env")


## A 部分：文档摄取与分块

从 `knowledge-base` 加载员工 / 产品 / 合同 / 公司等 Markdown，再切成适合嵌入的文本块（chunks）。


In [ ]:
# ========== 从 knowledge-base 各子文件夹加载 Markdown ==========
# 每个子文件夹名将写入 metadata["doc_type"]
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    # 例如 employees / products / contracts / company
    doc_type = os.path.basename(folder)
    # 递归加载该文件夹下全部 .md，强制 utf-8
    loader = DirectoryLoader(
        folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        # 给每篇打上类型标签，后面可视化按颜色区分
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")


In [ ]:
# ========== 用 RecursiveCharacterTextSplitter 分块（第 2 天风格）==========
# chunk_size=1000：单块目标长度；chunk_overlap=200：相邻块重叠，减轻边界截断
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")
# 预览首块前 300 字符，肉眼检查切分是否合理
print(f"Sample chunk:\n{chunks[0].page_content[:300]}...")


## B 部分：向量存储（Chroma）

把文本块编码成向量并写入 Chroma。默认用免费的 HuggingFace `all-MiniLM-L6-v2`；若要更高质量可改用 OpenAI 嵌入（见代码注释行）。


In [ ]:
# ========== 构建 HuggingFace 嵌入 +（重建）Chroma 向量库 ==========
# 免费本地小模型；首次会下载权重
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# 备选：OpenAI 嵌入（更高质量、需额度）——保持注释，勿改标识符/字符串
# 嵌入 = OpenAIEmbeddings(model="text-embedding-3-large")

# 若目录已存在旧库，先删 collection，避免新旧向量混在一起
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

# from_documents：分块 → 嵌入 → 写入 persist_directory
vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=DB_NAME
)
print(f"Vector store created with {vectorstore._collection.count()} documents")


### 可选：可视化向量（第 2 天风格）

用 t-SNE 把高维嵌入降到 2D，并按 `doc_type` 着色，直观检查不同类别文档是否在向量空间成簇。


In [ ]:
# ========== t-SNE 2D 可视化：观察向量库是否按文档类型成簇 ==========
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# 取出底层 Chroma collection
collection = vectorstore._collection
# 同时拉回 embeddings / documents / metadatas
result = collection.get(include=["embeddings", "documents", "metadatas"])
# 变成 (n_samples, dim) 的 numpy 矩阵
vectors = np.array(result["embeddings"])
# 从 metadata 取类型；缺失则 unknown
doc_types = [m.get("doc_type", "unknown") for m in result["metadatas"]]
# 类别 → 颜色映射
colors = {"products": "blue", "employees": "green", "contracts": "red", "company": "orange"}
color_list = [colors.get(t, "gray") for t in doc_types]

# random_state 固定，保证可复现
tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(vectors)
# Plotly 散点：悬停显示 doc_type
fig = go.Figure(data=[go.Scatter(
    x=reduced[:, 0], y=reduced[:, 1], mode="markers",
    marker=dict(size=5, color=color_list, opacity=0.8),
    text=doc_types, hoverinfo="text"
)])
fig.update_layout(title="2D Vector Store Visualization", width=700, height=500)
fig.show()


## C 部分：RAG 管道（第 3 天）

把 Retriever 与 ChatOpenAI 接起来：先检索相关块拼进 System Prompt，再让模型基于上下文回答。


In [ ]:
# ========== RAG：Retriever + 带上下文的 ChatOpenAI ==========
# k=10：每次检索最多取 10 个相关块
retriever = vectorstore.as_retriever(k=10)
# temperature=0：更偏确定性；模型名用前面的 MODEL 常量
llm = ChatOpenAI(temperature=0, model_name=MODEL)

# System Prompt 模板：要求只依据 Context；英文原文勿改（影响模型行为）
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing Insurellm.
Use the given context to answer questions. If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history=None):
    # history 默认空列表，避免可变默认参数陷阱
    history = history or []
    # 按问题检索相关 Document
    docs = retriever.invoke(question)
    # 把各块正文用空行拼成一大段上下文
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    # 先放 system，再回放 history 里的 user 消息，最后追加当前问题
    messages = [SystemMessage(content=system_prompt)]
    for h in history:
        if h.get("role") == "user":
            messages.append(HumanMessage(content=h["content"]))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    # 同时返回答案文本与检索到的 docs，便于调试/评估
    return response.content, docs


In [ ]:
# ========== 冒烟测试：问一个应能从知识库答出的事实题 ==========
answer, docs = answer_question("Who founded Insurellm?")
print("Q: Who founded Insurellm?")
print(f"A: {answer}")
# 看检索召回了多少块
print(f"\nRetrieved {len(docs)} chunks")


## D 部分：评估（第 4 天）

对测试集中的少量问题做答案抽检。完整评估模块（`evaluation.eval`）通常依赖 `implementation.answer`——若要跑官方全套，请先在 week5 目录执行 `implementation/ingest.py`。这里直接复用本笔记本的 `answer_question`。


In [ ]:
# ========== 从 evaluation 套件抽 3 题，对照参考答案肉眼检查 ==========
from evaluation.test import load_tests

tests = load_tests()
# 只跑前 3 题，控制耗时与 API 费用
sample = tests[:3]  # First 3 tests

for t in sample:
    answer, docs = answer_question(t.question)
    print(f"Q: {t.question}")
    print(f"A: {answer}")
    # 打印参考答案，方便并排比较
    print(f"Reference: {t.reference_answer}")
    print("-" * 60)


## E 部分：Gradio 聊天界面

为 Insurellm 专家助理提供交互式问答 UI；历史消息会转成 `answer_question` 需要的 dict 列表。


In [ ]:
# ========== Gradio ChatInterface：兼容 messages / 元组两种 history 格式 ==========

def chat_fn(message, history):
    history = history or []
    # 将 Gradio 历史转换为 [{"role","content"}, ...]，供 answer_question 回放
    prior = []
    for h in history:
        if isinstance(h, dict):
            # 新版 messages 格式
            prior.append({"role": h.get("role", "user"), "content": h.get("content", str(h))})
        elif isinstance(h, (list, tuple)) and len(h) >= 2:
            # 旧版 (user, assistant) 元组格式
            prior.append({"role": "user", "content": h[0]})
            prior.append({"role": "assistant", "content": h[1]})
    # 只要答案文本；检索 docs 用 _ 丢弃
    answer, _ = answer_question(message, prior)
    return answer

# type="messages"：与新版 Gradio 聊天消息格式对齐；title 保持英文原样
gr.ChatInterface(chat_fn, title="Insurellm Expert Assistant", type="messages").launch(inbrowser=True)
